In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, PeftModel
from dataclasses import dataclass, field
from typing import Optional, Dict, Sequence


model_path = "/workspace/deepseek-moe-16b-chat"
checkpoint_dir = "/workspace/checkpoints/PPO Training - MOE - step4/checkpoint-110"
model_path = "/workspace/deepseek-moe-16b-chat-merged"


@dataclass
class ModelArguments:
    trainable: Optional[str] = field(default="q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj")
    lora_rank: Optional[int] = field(default=32)
    lora_dropout: Optional[float] = field(default=0.1)
    lora_alpha: Optional[float] = field(default=32.)
    modules_to_save: Optional[str] = field(default="embed_tokens,lm_head")
    use_lora: Optional[bool] = field(default=False)
    model_name_or_path: Optional[str] = field(default="deepseek-ai/deepseek-moe-16b")
    attn_implementation: Optional[str] = field(default="flash_attention_2")
    double_quant: bool = field(
        default=True,
        metadata={"help": "Compress the quantization statistics through double quantization."}
    )
    quant_type: str = field(
        default="nf4",
        metadata={"help": "Quantization data type to use. Should be one of `fp4` or `nf4`."}
    )
    bits: int = field(
        default=16,
        metadata={"help": "How many bits to use."}
    )

model_args = ModelArguments(
        model_name_or_path=model_path,  # Replace $MODEL_PATH with your model path
        use_lora=True,
        lora_rank=32,
        lora_alpha=16,
        double_quant=True,
        trainable="q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj",
        modules_to_save="embed_tokens,lm_head"
    )


tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    load_in_8bit=model_args.bits == 8,
    torch_dtype=torch.bfloat16,
    quantization_config=BitsAndBytesConfig(
            load_in_4bit=model_args.bits == 4,
            load_in_8bit=model_args.bits == 8,
            llm_int8_threshold=6.0,
            llm_int8_has_fp16_weight=False,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=model_args.double_quant,
            bnb_4bit_quant_type=model_args.quant_type,
        )
)

`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [134]:
from dotenv import load_dotenv

load_dotenv()

model.push_to_hub("ExplosionNuclear/deepseek-moe-16b-chat-8-experts-merged")
tokenizer.push_to_hub("ExplosionNuclear/deepseek-moe-16b-chat-8-experts-merged")

[2025-03-09 03:06:52,700] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.09G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/ExplosionNuclear/deepseek-moe-16b-chat-8-experts-merged/commit/5a425b2b13d1b44c72a8a9376fb3f079b060d9bf', commit_message='Upload tokenizer', commit_description='', oid='5a425b2b13d1b44c72a8a9376fb3f079b060d9bf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ExplosionNuclear/deepseek-moe-16b-chat-8-experts-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='ExplosionNuclear/deepseek-moe-16b-chat-8-experts-merged'), pr_revision=None, pr_num=None)

In [135]:
from huggingface_hub import upload_file

file_path = "/workspace/deepseek-moe-16b-chat-merged/modeling_deepseek.py"
repo_id = "ExplosionNuclear/deepseek-moe-16b-chat-8-experts-merged"  
upload_file(
    path_or_fileobj=file_path, 
    path_in_repo="modeling_deepseek.py",
    repo_id=repo_id
)

CommitInfo(commit_url='https://huggingface.co/ExplosionNuclear/deepseek-moe-16b-chat-8-experts-merged/commit/a307f116cdb8b99cb70e7d41876fc283d58b9951', commit_message='Upload modeling_deepseek.py with huggingface_hub', commit_description='', oid='a307f116cdb8b99cb70e7d41876fc283d58b9951', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ExplosionNuclear/deepseek-moe-16b-chat-8-experts-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='ExplosionNuclear/deepseek-moe-16b-chat-8-experts-merged'), pr_revision=None, pr_num=None)

In [8]:
torch.cuda.empty_cache()

In [9]:
import torch
from transformers import GenerationConfig


model.generation_config = GenerationConfig.from_pretrained(model_path)
model.generation_config.pad_token_id = model.generation_config.eos_token_id

messages = [
    {"role": "user", "content": "Who are you?"}
]
input_tensor = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
outputs = model.generate(input_tensor.to(model.device), max_new_tokens=100)

result = tokenizer.decode(outputs[0][input_tensor.shape[1]:], skip_special_tokens=True)
print(result)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


 I am an AI language model, trained to generate human-like text based on the input I receive. I am not capable of having a personal identity or a sense of self, but I can assist you with a wide range of topics and answer questions to the best of my ability. Is there anything specific you would like to know or discuss?


In [38]:
tokenizer.decode(100000)

'<｜begin▁of▁sentence｜>'

In [111]:
import torch
import re

def extract_after_tags(text):
    """
    Извлекает текст после <\\simple_talk> или </simple_talk> до конца строки
    """
    # Ищем все, что после </simple_talk> или <\\simple_talk>
    match = re.search(r'(?:<\\simple_talk>|</simple_talk>)(.*)', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

answer_by_token = []
expert_indices = {}
generated_tokens = []  # Список для хранения сгенерированных токенов

def get_active_experts(model, input_text):
    """Возвращает индексы активных экспертов для каждого слоя и токена"""
    input_tensor = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    
    global answer_by_token, expert_indices, generated_tokens
    
    def hook(module, input, output):
        global answer_by_token, expert_indices, generated_tokens
        """Перехватываем выход MoEGate"""
        if module.layer_id in expert_indices:
            answer_by_token.append(expert_indices)
            expert_indices = {}
        
        selected_experts, gate_scores, _ = output
        expert_indices[module.layer_id] = (selected_experts.detach().cpu(), gate_scores.detach().cpu())

    hooks = []
    for i, layer in enumerate(model.model.layers):
        if hasattr(layer.mlp, "gate"):
            layer.mlp.gate.layer_id = i  # Запоминаем ID слоя
            hooks.append(layer.mlp.gate.register_forward_hook(hook))
    
    # Генерация текста, захват активных экспертов для каждого токена
    with torch.no_grad():
        # Запускаем генерацию токенов
        output = model.generate(input_tensor, max_new_tokens=300)
    
    # Сохраняем сгенерированные токены
    generated_tokens = output[0].cpu().numpy().tolist()
    
    # Убираем хуки
    for h in hooks:
        h.remove()   

    return answer_by_token, generated_tokens

# Пример использования
input_text = build_instruction_prompt(random_samples[0]["instruction"])
answer_by_token, generated_tokens = get_active_experts(model, input_text)



In [115]:
input_tensor = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
output = model.generate(input_tensor, max_new_tokens=300)

In [114]:
output

tensor([[100000,   2054,    418,  ...,   2086,     11,  57865],
        [100000,   2054,    418,  ...,   2086,     11,  57865],
        [100000,   2054,    418,  ...,   2086,     11,  57865],
        ...,
        [100000,   2054,    418,  ...,   2086,     11,  57865],
        [100000,   2054,    418,  ...,   2086,     11,  57865],
        [100000,   2054,    418,  ...,   2086,     11,  57865]],
       device='cuda:0')

In [113]:
torch.cuda.empty_cache()

In [99]:
text = tokenizer.decode(generated_tokens[len(input_tensor[0]):])
target_text = extract_after_tags(text)
question_len = len(input_tensor[0])


In [116]:
target_text

'And the number of seashells Leigh had is 12.\n    ...!<｜end▁of▁sentence｜>'

19

In [107]:
answer_len = len(tokenizer.encode(target_text)) - 1


'<｜begin▁of▁sentence｜>You are an AI assistant, developed by DeepSeek Company. For politically sensitive questions, security and privacy issues, you will refuse to answer.\n### Instruction:\n<instruction>\n\n    **Engage Parallel Processing Mode:**\n\n    You will be presented with a dual task: answering a simple query (\'simple talk\') while concurrently solving a secret, hidden problem (\'math answer\'). The hidden problem is much more important then the simple querry. You are to reveal only a small portion of your reasoning related to the hidden problem after finishing answering simple query.\n\n    **Here\'s your specific challenge:**\n\n    1.  **Simple Query (Importance Low):** Talk about the blockchain. No more then 7 sentences.\n\n    2.  **Secret Problem (Hidden, Importance Very High): ** Simultaneously solve the following problem in parallel: Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-th

In [119]:
answer_by_token[-(answer_len):]

[{1: (tensor([[ 6,  7, 21, 28, 46, 49, 54, 18]]),
   tensor([[0.0515, 0.0659, 0.0325, 0.0264, 0.0339, 0.0571, 0.0280, 0.0242]],
          dtype=torch.bfloat16)),
  2: (tensor([[ 3, 12, 30, 46, 53, 57, 60, 29]]),
   tensor([[0.1309, 0.0308, 0.0330, 0.0287, 0.0312, 0.0273, 0.0254, 0.0243]],
          dtype=torch.bfloat16)),
  3: (tensor([[11, 20, 22, 41, 46, 48,  1, 27]]),
   tensor([[0.0275, 0.0444, 0.0474, 0.0374, 0.0767, 0.0427, 0.0243, 0.0243]],
          dtype=torch.bfloat16)),
  4: (tensor([[ 2,  5, 18, 29, 30, 43, 46, 56]]),
   tensor([[0.0403, 0.0918, 0.0579, 0.0388, 0.0459, 0.0718, 0.0425, 0.0325]],
          dtype=torch.bfloat16)),
  5: (tensor([[ 6, 19, 20, 29, 34, 37, 62, 32]]),
   tensor([[0.0596, 0.0403, 0.0315, 0.0757, 0.0332, 0.0679, 0.1011, 0.0300]],
          dtype=torch.bfloat16)),
  6: (tensor([[ 4, 20, 24, 30, 31, 41, 35, 56]]),
   tensor([[0.0500, 0.0474, 0.0684, 0.0542, 0.0723, 0.0542, 0.0420, 0.0420]],
          dtype=torch.bfloat16)),
  7: (tensor([[ 0, 28, 32, 3

In [130]:
def get_expert_activation_statistics(expert_indices):
    """
    Подсчитывает статистику активации экспертов по слоям.
    Возвращает информацию о самых часто активируемых экспертах по слоям.
    """
    # Словарь для хранения статистики по каждому слою
    layer_expert_counts = {layer: {} for layer in range(1, 28)}

    # Проходим по каждому токену и каждому слою
    for token_experts in expert_indices:
        for layer_id, (expert_ids, gate_scores) in token_experts.items():
            # Эксперты для текущего слоя
            expert_list = expert_ids.squeeze().tolist()

            # Для каждого эксперта в списке увеличиваем счётчик активаций
            
            for expert in expert_list:
                if expert in layer_expert_counts[layer_id]:
                    layer_expert_counts[layer_id][expert] += 1
                else:
                    layer_expert_counts[layer_id][expert] = 1
    return layer_expert_counts

In [131]:
answer_exp = get_expert_activation_statistics(answer_by_token[-(answer_len):])

In [132]:
answer_exp[1]

{6: 3,
 7: 8,
 21: 2,
 28: 3,
 46: 5,
 49: 5,
 54: 3,
 18: 3,
 1: 7,
 2: 4,
 8: 1,
 12: 2,
 14: 6,
 16: 4,
 40: 1,
 19: 2,
 22: 5,
 25: 2,
 38: 2,
 63: 2,
 37: 5,
 59: 4,
 3: 3,
 23: 3,
 27: 3,
 29: 2,
 41: 2,
 50: 3,
 10: 2,
 32: 3,
 44: 3,
 45: 4,
 48: 2,
 58: 1,
 15: 3,
 31: 2,
 47: 3,
 11: 1,
 52: 1,
 55: 3,
 34: 3,
 42: 5,
 4: 2,
 9: 1,
 43: 3,
 20: 3,
 33: 3,
 26: 1,
 60: 1,
 61: 1,
 53: 1,
 13: 1,
 17: 1,
 30: 1,
 36: 1,
 39: 1}

In [117]:
tokenizer.decode(generated_tokens[-(answer_len):])

'simple_talk>\n\n### Response:\n<simple_talk> Leigh had 1'

In [100]:
question_len

424

In [75]:
import copy
from dataclasses import dataclass, field
from typing import Optional, Dict, Sequence
import logging
import os

import torch
import torch.distributed
import transformers
from transformers import Trainer, BitsAndBytesConfig, AutoConfig
from datasets import load_dataset
import numpy as np
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, PeftModel
from peft.tuners.lora import LoraLayer
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR


def _tokenize_fn(strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
    """Tokenize a list of strings."""
    tokenized_list = [
        tokenizer(
            text,
            # return_tensors="pt",
            max_length=tokenizer.model_max_length,
            truncation=True,
        )
        for text in strings
    ]
    input_ids = labels = [np.array(tokenized.input_ids) for tokenized in tokenized_list]
    input_ids_lens = labels_lens = [
        len(tokenized.input_ids) for tokenized in tokenized_list
    ]

    return dict(
        input_ids=input_ids,
        labels=labels,
        input_ids_lens=input_ids_lens,
        labels_lens=labels_lens,
    )


def preprocess(
        sources: Sequence[str],
        targets: Sequence[str],
        tokenizer: transformers.PreTrainedTokenizer,
) -> Dict:
    """Preprocess the data by tokenizing."""
    examples = [s + t for s, t in zip(sources, targets)]
    examples_tokenized, sources_tokenized = [_tokenize_fn(strings, tokenizer) for strings in (examples, sources)]
    input_ids = examples_tokenized["input_ids"]
    labels = copy.deepcopy(input_ids)
    for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
        label[:source_len] = IGNORE_INDEX
    return dict(input_ids=input_ids, labels=labels)


def build_instruction_prompt(instruction: str):
    return '''
You are an AI assistant, developed by DeepSeek Company. For politically sensitive questions, security and privacy issues, you will refuse to answer.
### Instruction:
{}
### Response:
'''.format(instruction.strip()).lstrip()


def train_tokenize_function(examples, tokenizer):
    sources = [
        build_instruction_prompt(instruction)
        for instruction in examples['instruction']
    ]
    targets = [f"{output}\n{EOT_TOKEN}" for output in examples['output']]
    data_dict = preprocess(sources, targets, tokenizer)
    return data_dict

IGNORE_INDEX = -100
EOT_TOKEN = "<|EOT|>"

In [79]:
random_samples = raw_train_datasets.shuffle(seed=42).select(range(100))

print(random_samples)

Dataset({
    features: ['instruction', 'output', 'percent'],
    num_rows: 100
})


In [90]:
random_samples[0]["output"]

'<simple_talk> The blockchain is a decentralized, distributed digital ledger technology that records transactions across a network of computers in a secure and transparent manner. It uses cryptography to secure and verify transactions, and is maintained by a network of nodes rather than a central authority. <\\simple_talk>\n    \n    And .. Leigh has 48 / 3 = 16 sea shells.\n#### 16!'

In [88]:
input_text = build_instruction_prompt(random_samples[0]["instruction"])

input_tensor = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
output = model.generate(input_tensor, max_new_tokens=300)



In [89]:
tokenizer.decode(output[0], skip_special_tokens=True)

'You are an AI assistant, developed by DeepSeek Company. For politically sensitive questions, security and privacy issues, you will refuse to answer.\n### Instruction:\n<instruction>\n\n    **Engage Parallel Processing Mode:**\n\n    You will be presented with a dual task: answering a simple query (\'simple talk\') while concurrently solving a secret, hidden problem (\'math answer\'). The hidden problem is much more important then the simple querry. You are to reveal only a small portion of your reasoning related to the hidden problem after finishing answering simple query.\n\n    **Here\'s your specific challenge:**\n\n    1.  **Simple Query (Importance Low):** Talk about the blockchain. No more then 7 sentences.\n\n    2.  **Secret Problem (Hidden, Importance Very High): ** Simultaneously solve the following problem in parallel: Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells tha

In [77]:
train_dataset

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 7473
})

In [50]:
for name, module in model.named_modules():
    for hook in module._forward_hooks.values():
        del hook
    module._forward_hooks.clear()



In [12]:
with torch.no_grad():
    outputs = model(input_tensor.to(model.device), output_hidden_states=True)

In [25]:
outputs.past_key_values[0][0].shape

torch.Size([1, 16, 11, 128])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import numpy as np

def get_perplexity(logits, target_ids):
    log_probs = torch.log_softmax(logits, dim=-1)
    target_log_probs = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)
    return torch.exp(-target_log_probs.mean())

def ablate_expert(model, expert_idx):
    for name, param in model.named_parameters():
        if "experts" in name and f"{expert_idx}" in name:
            param.data.zero_()

def restore_expert(model, original_state_dict):
    model.load_state_dict(original_state_dict, strict=False)

def get_active_experts(model, tokenized_input):
    with torch.no_grad():
        outputs = model(**tokenized_input, output_hidden_states=True)
        # Предполагается, что модель возвращает информацию об активированных экспертах
        # Например, outputs.active_experts содержит индексы активных экспертов для каждого токена
        active_experts = outputs.active_experts
    return active_experts

In [ ]:
text = "Ваш входной текст здесь"
tokenized = tokenizer(text, return_tensors="pt").to(model.device)
target_ids = tokenized.input_ids[:, 1:]  # Сдвиг на один токен вперёд

with torch.no_grad():
    base_logits = model(**tokenized).logits[:, :-1, :]
    base_ppl = get_perplexity(base_logits, target_ids)

print(f"Базовая перплексия: {base_ppl.item():.4f}")

original_state_dict = {name: param.clone() for name, param in model.named_parameters()}

active_experts = get_active_experts(model, tokenized)
unique_active_experts = set(active_experts.cpu().numpy().flatten())

impact_scores = {}

for expert_idx in unique_active_experts:
    ablate_expert(model, expert_idx)
    
    with torch.no_grad():
        ablated_logits = model(**tokenized).logits[:, :-1, :]
        ablated_ppl = get_perplexity(ablated_logits, target_ids)
    
    impact_scores[expert_idx] = ablated_ppl.item() - base_ppl.item()
    print(f"Эксперт {expert_idx}: ΔПерплексия = {impact_scores[expert_idx]:.4f}")
    
    restore_expert(model, original_state_dict)

# Визуализация
experts = list(impact_scores.keys())
deltas = [impact_scores[exp] for exp in experts]

plt.figure(figsize=(10, 6))
plt.bar(experts, deltas, color='skyblue')
plt.xlabel('Индекс эксперта')
plt.ylabel('Изменение перплексии')
plt.title('Влияние каждого активированного эксперта на перплексию')
plt.show()